# HPC Lab – Experiment 11
## CUDA Performance Optimization and cuBLAS

**Name:** Aditi Mittal | **PRN:** 23610067 | **Batch:** B1  
**Course:** Final Year B.Tech CSE | **College:** Walchand College of Engineering

---
### Instructions
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Run all cells top-to-bottom (Runtime → Run all)
3. Take screenshots of the outputs for your report

**Topics covered:**
- Task 1: CPU baseline → Basic CUDA (global memory) → Tiled CUDA (shared memory)
- Task 2: cuBLAS vs hand-written CUDA kernels
- All-in-one benchmark with 3 summary tables + performance graphs

## Step 0 – Verify GPU is available

In [ ]:
!nvidia-smi

---
## Task 1 – Step 1: CPU Sequential Matrix Multiplication

Baseline: ikj loop order (cache-friendly), sizes 256×256, 512×512, 1024×1024.  
Reports execution time (ms) and GFLOPS.

In [ ]:
%%writefile 01_cpu_matmul.cpp
#include <stdio.h>
#include <stdlib.h>
#include <chrono>
#include <math.h>

// ikj loop order — better cache behaviour
void matmul_cpu(const float* A, const float* B, float* C, int N) {
    for (int i = 0; i < N; i++) {
        for (int k = 0; k < N; k++) {
            float a_ik = A[i*N + k];
            for (int j = 0; j < N; j++)
                C[i*N + j] += a_ik * B[k*N + j];
        }
    }
}

int main() {
    int sizes[]   = {256, 512, 1024};
    int num_sizes = 3;

    printf("\n=== CPU Sequential Matrix Multiplication ===\n");
    printf("%-14s  %-16s  %-10s\n", "Matrix Size", "CPU Time (ms)", "GFLOPS");
    printf("%-14s  %-16s  %-10s\n", "----------",  "-------------", "------");

    for (int s = 0; s < num_sizes; s++) {
        int N = sizes[s];
        size_t bytes = (size_t)N * N * sizeof(float);

        float* A = (float*)malloc(bytes);
        float* B = (float*)malloc(bytes);
        float* C = (float*)calloc(N * N, sizeof(float));

        srand(42);
        for (int i = 0; i < N*N; i++) {
            A[i] = (float)rand() / RAND_MAX;
            B[i] = (float)rand() / RAND_MAX;
        }

        auto t0 = std::chrono::high_resolution_clock::now();
        matmul_cpu(A, B, C, N);
        auto t1 = std::chrono::high_resolution_clock::now();

        double ms     = std::chrono::duration<double, std::milli>(t1 - t0).count();
        double flops  = 2.0 * N * N * N;
        double gflops = flops / (ms * 1e6);

        printf("%-14d  %-16.2f  %-10.4f\n", N, ms, gflops);

        free(A); free(B); free(C);
    }
    printf("\n");
    return 0;
}

In [ ]:
!g++ -O2 -o cpu_matmul 01_cpu_matmul.cpp && ./cpu_matmul

---
## Task 1 – Step 2: Basic CUDA Matrix Multiplication (Global Memory)

Each CUDA thread computes one output element, reading from global memory.  
Tests block sizes 8×8 and 16×16. Reports kernel-only time AND total time (H2D + kernel + D2H).

In [ ]:
%%writefile 02_cuda_basic.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <math.h>

#define CUDA_CHECK(err)                                                        \
    do {                                                                       \
        cudaError_t _e = (err);                                                \
        if (_e != cudaSuccess) {                                               \
            fprintf(stderr, "CUDA error %s at %s:%d\n",                       \
                    cudaGetErrorString(_e), __FILE__, __LINE__);               \
            exit(1);                                                           \
        }                                                                      \
    } while (0)

// One thread per output element, all reads from global memory
__global__ void matmul_basic_kernel(const float* __restrict__ A,
                                    const float* __restrict__ B,
                                    float* C, int N)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++)
            sum += A[row*N + k] * B[k*N + col];
        C[row*N + col] = sum;
    }
}

void benchmark_basic(float* d_A, float* d_B, float* d_C,
                     float* h_A, float* h_B, float* h_C,
                     int N, int BS)
{
    size_t bytes = (size_t)N * N * sizeof(float);
    dim3 block(BS, BS);
    dim3 grid((N + BS - 1) / BS, (N + BS - 1) / BS);

    cudaEvent_t ev_ts, ev_te, ev_ks, ev_ke;
    cudaEventCreate(&ev_ts); cudaEventCreate(&ev_te);
    cudaEventCreate(&ev_ks); cudaEventCreate(&ev_ke);

    cudaEventRecord(ev_ts);
    CUDA_CHECK(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_C, 0, bytes));

    cudaEventRecord(ev_ks);
    matmul_basic_kernel<<<grid, block>>>(d_A, d_B, d_C, N);
    cudaEventRecord(ev_ke);

    CUDA_CHECK(cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost));
    cudaEventRecord(ev_te);
    cudaEventSynchronize(ev_te);

    float kern_ms = 0, total_ms = 0;
    cudaEventElapsedTime(&kern_ms,  ev_ks, ev_ke);
    cudaEventElapsedTime(&total_ms, ev_ts, ev_te);

    double gflops = (2.0 * N * N * N) / (kern_ms * 1e6);
    printf("%-14d  %-12d  %-18.4f  %-18.4f  %-10.4f\n",
           N, BS, kern_ms, total_ms, gflops);

    cudaEventDestroy(ev_ts); cudaEventDestroy(ev_te);
    cudaEventDestroy(ev_ks); cudaEventDestroy(ev_ke);
}

int main()
{
    int sizes[]       = {256, 512, 1024};
    int block_sizes[] = {8, 16};

    printf("\n=== Basic CUDA Matrix Multiplication (Global Memory) ===\n");
    printf("%-14s  %-12s  %-18s  %-18s  %-10s\n",
           "Matrix Size", "Block Size", "Kernel Time (ms)", "Total Time (ms)", "GFLOPS");
    printf("%-14s  %-12s  %-18s  %-18s  %-10s\n",
           "-----------", "----------", "----------------", "---------------", "------");

    for (int s = 0; s < 3; s++) {
        int N     = sizes[s];
        size_t sz = (size_t)N * N * sizeof(float);

        float *h_A = (float*)malloc(sz);
        float *h_B = (float*)malloc(sz);
        float *h_C = (float*)malloc(sz);

        srand(42);
        for (int i = 0; i < N*N; i++) {
            h_A[i] = (float)rand() / RAND_MAX;
            h_B[i] = (float)rand() / RAND_MAX;
        }

        float *d_A, *d_B, *d_C;
        CUDA_CHECK(cudaMalloc(&d_A, sz));
        CUDA_CHECK(cudaMalloc(&d_B, sz));
        CUDA_CHECK(cudaMalloc(&d_C, sz));

        for (int b = 0; b < 2; b++)
            benchmark_basic(d_A, d_B, d_C, h_A, h_B, h_C, N, block_sizes[b]);

        free(h_A); free(h_B); free(h_C);
        cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    }
    printf("\n");
    return 0;
}

In [ ]:
!nvcc -O2 -o cuda_basic 02_cuda_basic.cu && ./cuda_basic

---
## Task 1 – Step 3: Tiled CUDA Matrix Multiplication (Shared Memory)

Key idea: each thread block loads a tile of A and B into shared memory (fast SRAM),  
then reuses those values across the dot-product — reduces global memory traffic by TILE_SIZE×.  
Tests tile sizes 8×8 and 16×16.

In [ ]:
%%writefile 03_cuda_tiled.cu
// Each thread block loads a TILE_SIZE×TILE_SIZE tile of A and B into shared memory,
// accumulates the dot product, then moves to the next tile — cuts global memory
// traffic by TILE_SIZE×.
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <math.h>

#define CUDA_CHECK(err)                                                        \
    do {                                                                       \
        cudaError_t _e = (err);                                                \
        if (_e != cudaSuccess) {                                               \
            fprintf(stderr, "CUDA error %s at %s:%d\n",                       \
                    cudaGetErrorString(_e), __FILE__, __LINE__);               \
            exit(1);                                                           \
        }                                                                      \
    } while (0)

template<int TILE_SIZE>
__global__ void matmul_tiled_kernel(const float* __restrict__ A,
                                    const float* __restrict__ B,
                                    float* C, int N)
{
    __shared__ float sA[TILE_SIZE][TILE_SIZE];
    __shared__ float sB[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;
    float sum = 0.0f;

    for (int t = 0; t < (N + TILE_SIZE - 1) / TILE_SIZE; t++) {
        int a_col = t * TILE_SIZE + threadIdx.x;
        int b_row = t * TILE_SIZE + threadIdx.y;
        sA[threadIdx.y][threadIdx.x] = (row < N && a_col < N) ? A[row*N + a_col] : 0.0f;
        sB[threadIdx.y][threadIdx.x] = (b_row < N && col < N) ? B[b_row*N + col] : 0.0f;
        __syncthreads();

        #pragma unroll
        for (int k = 0; k < TILE_SIZE; k++)
            sum += sA[threadIdx.y][k] * sB[k][threadIdx.x];
        __syncthreads();
    }

    if (row < N && col < N)
        C[row*N + col] = sum;
}

template<int TILE_SIZE>
void benchmark_tiled(float* d_A, float* d_B, float* d_C,
                     float* h_A, float* h_B, float* h_C, int N)
{
    size_t bytes = (size_t)N * N * sizeof(float);
    dim3 block(TILE_SIZE, TILE_SIZE);
    dim3 grid((N + TILE_SIZE - 1) / TILE_SIZE, (N + TILE_SIZE - 1) / TILE_SIZE);

    cudaEvent_t ev_ts, ev_te, ev_ks, ev_ke;
    cudaEventCreate(&ev_ts); cudaEventCreate(&ev_te);
    cudaEventCreate(&ev_ks); cudaEventCreate(&ev_ke);

    cudaEventRecord(ev_ts);
    CUDA_CHECK(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_C, 0, bytes));

    cudaEventRecord(ev_ks);
    matmul_tiled_kernel<TILE_SIZE><<<grid, block>>>(d_A, d_B, d_C, N);
    cudaEventRecord(ev_ke);

    CUDA_CHECK(cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost));
    cudaEventRecord(ev_te);
    cudaEventSynchronize(ev_te);

    float kern_ms = 0, total_ms = 0;
    cudaEventElapsedTime(&kern_ms,  ev_ks, ev_ke);
    cudaEventElapsedTime(&total_ms, ev_ts, ev_te);

    double gflops = (2.0 * N * N * N) / (kern_ms * 1e6);
    printf("%-14d  %-12d  %-18.4f  %-18.4f  %-10.4f\n",
           N, TILE_SIZE, kern_ms, total_ms, gflops);

    cudaEventDestroy(ev_ts); cudaEventDestroy(ev_te);
    cudaEventDestroy(ev_ks); cudaEventDestroy(ev_ke);
}

int main()
{
    int sizes[] = {256, 512, 1024};

    printf("\n=== Tiled CUDA Matrix Multiplication (Shared Memory) ===\n");
    printf("%-14s  %-12s  %-18s  %-18s  %-10s\n",
           "Matrix Size", "Tile Size", "Kernel Time (ms)", "Total Time (ms)", "GFLOPS");
    printf("%-14s  %-12s  %-18s  %-18s  %-10s\n",
           "-----------", "---------", "----------------", "---------------", "------");

    for (int s = 0; s < 3; s++) {
        int N     = sizes[s];
        size_t sz = (size_t)N * N * sizeof(float);

        float *h_A = (float*)malloc(sz);
        float *h_B = (float*)malloc(sz);
        float *h_C = (float*)malloc(sz);

        srand(42);
        for (int i = 0; i < N*N; i++) {
            h_A[i] = (float)rand() / RAND_MAX;
            h_B[i] = (float)rand() / RAND_MAX;
        }

        float *d_A, *d_B, *d_C;
        CUDA_CHECK(cudaMalloc(&d_A, sz));
        CUDA_CHECK(cudaMalloc(&d_B, sz));
        CUDA_CHECK(cudaMalloc(&d_C, sz));

        benchmark_tiled<8> (d_A, d_B, d_C, h_A, h_B, h_C, N);
        benchmark_tiled<16>(d_A, d_B, d_C, h_A, h_B, h_C, N);

        free(h_A); free(h_B); free(h_C);
        cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    }
    printf("\n");
    return 0;
}

In [ ]:
!nvcc -O2 -o cuda_tiled 03_cuda_tiled.cu && ./cuda_tiled

---
## Task 2: cuBLAS Matrix Multiplication with Correctness Verification

cuBLAS uses NVIDIA's auto-tuned kernels (register tiling, vectorised loads, warp-level optimisations).  
**Column-major trick:** pass B first, A second to `cublasSgemm` to handle row-major C arrays.  
Verifies correctness against CPU reference for sizes ≤512.

In [ ]:
%%writefile 04_cuda_cublas.cu
// Column-major trick: pass (d_B, d_A) to cublasSgemm so row-major C = A*B is computed correctly.
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>

#define CUDA_CHECK(err)                                                        \
    do {                                                                       \
        cudaError_t _e = (err);                                                \
        if (_e != cudaSuccess) {                                               \
            fprintf(stderr, "CUDA error %s at %s:%d\n",                       \
                    cudaGetErrorString(_e), __FILE__, __LINE__);               \
            exit(1);                                                           \
        }                                                                      \
    } while (0)

#define CUBLAS_CHECK(err)                                                      \
    do {                                                                       \
        cublasStatus_t _e = (err);                                             \
        if (_e != CUBLAS_STATUS_SUCCESS) {                                     \
            fprintf(stderr, "cuBLAS error %d at %s:%d\n",                     \
                    (int)_e, __FILE__, __LINE__);                              \
            exit(1);                                                           \
        }                                                                      \
    } while (0)

void matmul_cpu_ref(const float* A, const float* B, float* C, int N) {
    for (int i = 0; i < N; i++)
        for (int k = 0; k < N; k++) {
            float a = A[i*N + k];
            for (int j = 0; j < N; j++)
                C[i*N + j] += a * B[k*N + j];
        }
}

bool verify(const float* ref, const float* res, int N, float tol = 1e-3f) {
    for (int i = 0; i < N*N; i++) {
        float diff = fabsf(ref[i] - res[i]);
        float base = fmaxf(fabsf(ref[i]), 1e-6f);
        if (diff / base > tol) {
            printf("  Mismatch at index %d: ref=%.6f  got=%.6f\n", i, ref[i], res[i]);
            return false;
        }
    }
    return true;
}

int main()
{
    int sizes[] = {256, 512, 1024};

    cublasHandle_t handle;
    CUBLAS_CHECK(cublasCreate(&handle));

    printf("\n=== cuBLAS Matrix Multiplication ===\n");
    printf("%-14s  %-12s  %-18s  %-18s  %-10s\n",
           "Matrix Size", "Correct?", "Kernel Time (ms)", "Total Time (ms)", "GFLOPS");
    printf("%-14s  %-12s  %-18s  %-18s  %-10s\n",
           "-----------", "--------", "----------------", "---------------", "------");

    for (int s = 0; s < 3; s++) {
        int N     = sizes[s];
        size_t sz = (size_t)N * N * sizeof(float);

        float *h_A   = (float*)malloc(sz);
        float *h_B   = (float*)malloc(sz);
        float *h_C   = (float*)calloc(N*N, sizeof(float));
        float *h_ref = (float*)calloc(N*N, sizeof(float));

        srand(42);
        for (int i = 0; i < N*N; i++) {
            h_A[i] = (float)rand() / RAND_MAX;
            h_B[i] = (float)rand() / RAND_MAX;
        }

        bool checked = (N <= 512);
        if (checked) matmul_cpu_ref(h_A, h_B, h_ref, N);

        float *d_A, *d_B, *d_C;
        CUDA_CHECK(cudaMalloc(&d_A, sz));
        CUDA_CHECK(cudaMalloc(&d_B, sz));
        CUDA_CHECK(cudaMalloc(&d_C, sz));

        cudaEvent_t ev_ts, ev_te, ev_ks, ev_ke;
        cudaEventCreate(&ev_ts); cudaEventCreate(&ev_te);
        cudaEventCreate(&ev_ks); cudaEventCreate(&ev_ke);

        cudaEventRecord(ev_ts);
        CUDA_CHECK(cudaMemcpy(d_A, h_A, sz, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_B, h_B, sz, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemset(d_C, 0, sz));

        const float alpha = 1.0f, beta = 0.0f;
        cudaEventRecord(ev_ks);
        CUBLAS_CHECK(cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, N, N,
                                 &alpha, d_B, N, d_A, N, &beta, d_C, N));
        cudaEventRecord(ev_ke);

        CUDA_CHECK(cudaMemcpy(h_C, d_C, sz, cudaMemcpyDeviceToHost));
        cudaEventRecord(ev_te);
        cudaEventSynchronize(ev_te);

        float kern_ms = 0, total_ms = 0;
        cudaEventElapsedTime(&kern_ms,  ev_ks, ev_ke);
        cudaEventElapsedTime(&total_ms, ev_ts, ev_te);

        const char* correct_str = checked ? (verify(h_ref, h_C, N) ? "PASS" : "FAIL") : "N/A";
        double gflops = (2.0 * N * N * N) / (kern_ms * 1e6);

        printf("%-14d  %-12s  %-18.4f  %-18.4f  %-10.4f\n",
               N, correct_str, kern_ms, total_ms, gflops);

        free(h_A); free(h_B); free(h_C); free(h_ref);
        cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
        cudaEventDestroy(ev_ts); cudaEventDestroy(ev_te);
        cudaEventDestroy(ev_ks); cudaEventDestroy(ev_ke);
    }

    cublasDestroy(handle);
    printf("\n");
    return 0;
}

In [ ]:
!nvcc -O2 -o cuda_cublas 04_cuda_cublas.cu -lcublas && ./cuda_cublas

---
## All-in-One Benchmark (Recommended – run this for your report screenshots)

Runs all four implementations back-to-back:  
**CPU sequential → Basic CUDA → Tiled CUDA (16×16) → cuBLAS**  

Outputs:
- Table 1: Execution times (CPU vs CUDA variants)
- Table 2: cuBLAS vs CUDA with CPU speedup
- Table 3: GFLOPS for each implementation

In [ ]:
%%writefile 05_benchmark_all.cu
// All-in-one benchmark: CPU → Basic CUDA → Tiled CUDA (16×16) → cuBLAS
// Prints three summary tables for sizes 256, 512, 1024.
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <chrono>
#include <cuda_runtime.h>
#include <cublas_v2.h>

#define CUDA_CHECK(err)                                                        \
    do {                                                                       \
        cudaError_t _e = (err);                                                \
        if (_e != cudaSuccess) {                                               \
            fprintf(stderr, "CUDA error %s at %s:%d\n",                       \
                    cudaGetErrorString(_e), __FILE__, __LINE__);               \
            exit(1);                                                           \
        }                                                                      \
    } while (0)

#define CUBLAS_CHECK(err)                                                      \
    do {                                                                       \
        cublasStatus_t _e = (err);                                             \
        if (_e != CUBLAS_STATUS_SUCCESS) {                                     \
            fprintf(stderr, "cuBLAS error %d at %s:%d\n",                     \
                    (int)_e, __FILE__, __LINE__);                              \
            exit(1);                                                           \
        }                                                                      \
    } while (0)

void matmul_cpu(const float* A, const float* B, float* C, int N) {
    for (int i = 0; i < N; i++)
        for (int k = 0; k < N; k++) {
            float a = A[i*N + k];
            for (int j = 0; j < N; j++)
                C[i*N + j] += a * B[k*N + j];
        }
}

__global__ void kernel_basic(const float* A, const float* B, float* C, int N) {
    int r = blockIdx.y * blockDim.y + threadIdx.y;
    int c = blockIdx.x * blockDim.x + threadIdx.x;
    if (r < N && c < N) {
        float s = 0.0f;
        for (int k = 0; k < N; k++) s += A[r*N+k] * B[k*N+c];
        C[r*N+c] = s;
    }
}

template<int TS>
__global__ void kernel_tiled(const float* A, const float* B, float* C, int N) {
    __shared__ float sA[TS][TS], sB[TS][TS];
    int r = blockIdx.y*TS + threadIdx.y;
    int c = blockIdx.x*TS + threadIdx.x;
    float s = 0.0f;
    for (int t = 0; t < (N+TS-1)/TS; t++) {
        int ac = t*TS + threadIdx.x, br = t*TS + threadIdx.y;
        sA[threadIdx.y][threadIdx.x] = (r<N && ac<N) ? A[r*N+ac] : 0.0f;
        sB[threadIdx.y][threadIdx.x] = (br<N && c<N) ? B[br*N+c] : 0.0f;
        __syncthreads();
        #pragma unroll
        for (int k = 0; k < TS; k++) s += sA[threadIdx.y][k] * sB[k][threadIdx.x];
        __syncthreads();
    }
    if (r<N && c<N) C[r*N+c] = s;
}

float gpu_ms(cudaEvent_t a, cudaEvent_t b) {
    float ms; cudaEventElapsedTime(&ms, a, b); return ms;
}

bool ok(const float* ref, const float* res, int N) {
    for (int i = 0; i < N*N; i++) {
        float d = fabsf(ref[i] - res[i]) / fmaxf(fabsf(ref[i]), 1e-5f);
        if (d > 1e-3f) { printf("  MISMATCH at %d\n", i); return false; }
    }
    return true;
}

void print_device_info() {
    int dev; cudaGetDevice(&dev);
    cudaDeviceProp p; cudaGetDeviceProperties(&p, dev);
    printf("GPU  : %s\n", p.name);
    printf("SMs  : %d  |  Global mem: %.1f GB  |  Shared mem/block: %zu KB\n\n",
           p.multiProcessorCount, p.totalGlobalMem / 1e9, p.sharedMemPerBlock / 1024);
}

int main()
{
    printf("========================================\n");
    printf("  HPC Lab - Experiment 11 Benchmark\n");
    printf("  Name: Aditi Mittal  PRN: 23610067\n");
    printf("========================================\n\n");

    print_device_info();

    int sizes[] = {256, 512, 1024};
    const int TS = 16;

    double cpu_ms[3], basic_kern[3], basic_total[3];
    double tiled_kern[3], tiled_total[3];
    double cub_kern[3],   cub_total[3];

    cublasHandle_t handle;
    CUBLAS_CHECK(cublasCreate(&handle));

    for (int s = 0; s < 3; s++) {
        int N     = sizes[s];
        size_t sz = (size_t)N * N * sizeof(float);

        printf("--- N = %d ---\n", N);

        float *h_A   = (float*)malloc(sz);
        float *h_B   = (float*)malloc(sz);
        float *h_C   = (float*)calloc(N*N, sizeof(float));
        float *h_ref = (float*)calloc(N*N, sizeof(float));

        srand(42);
        for (int i = 0; i < N*N; i++) {
            h_A[i] = (float)rand() / RAND_MAX;
            h_B[i] = (float)rand() / RAND_MAX;
        }

        auto t0 = std::chrono::high_resolution_clock::now();
        matmul_cpu(h_A, h_B, h_ref, N);
        auto t1 = std::chrono::high_resolution_clock::now();
        cpu_ms[s] = std::chrono::duration<double,std::milli>(t1-t0).count();
        printf("  CPU: %.2f ms\n", cpu_ms[s]);

        float *d_A, *d_B, *d_C;
        CUDA_CHECK(cudaMalloc(&d_A, sz));
        CUDA_CHECK(cudaMalloc(&d_B, sz));
        CUDA_CHECK(cudaMalloc(&d_C, sz));

        cudaEvent_t e0,e1,e2,e3;
        cudaEventCreate(&e0); cudaEventCreate(&e1);
        cudaEventCreate(&e2); cudaEventCreate(&e3);

        // Basic CUDA
        {
            dim3 blk(TS,TS), grd((N+TS-1)/TS,(N+TS-1)/TS);
            cudaEventRecord(e0);
            CUDA_CHECK(cudaMemcpy(d_A, h_A, sz, cudaMemcpyHostToDevice));
            CUDA_CHECK(cudaMemcpy(d_B, h_B, sz, cudaMemcpyHostToDevice));
            CUDA_CHECK(cudaMemset(d_C, 0, sz));
            cudaEventRecord(e2);
            kernel_basic<<<grd,blk>>>(d_A, d_B, d_C, N);
            cudaEventRecord(e3);
            CUDA_CHECK(cudaMemcpy(h_C, d_C, sz, cudaMemcpyDeviceToHost));
            cudaEventRecord(e1); cudaEventSynchronize(e1);
            basic_kern[s]  = gpu_ms(e2,e3);
            basic_total[s] = gpu_ms(e0,e1);
            printf("  Basic CUDA: kern=%.4f ms  total=%.4f ms  correct=%s\n",
                   basic_kern[s], basic_total[s], ok(h_ref,h_C,N)?"YES":"NO");
        }

        // Tiled CUDA
        {
            dim3 blk(TS,TS), grd((N+TS-1)/TS,(N+TS-1)/TS);
            cudaEventRecord(e0);
            CUDA_CHECK(cudaMemcpy(d_A, h_A, sz, cudaMemcpyHostToDevice));
            CUDA_CHECK(cudaMemcpy(d_B, h_B, sz, cudaMemcpyHostToDevice));
            CUDA_CHECK(cudaMemset(d_C, 0, sz));
            cudaEventRecord(e2);
            kernel_tiled<TS><<<grd,blk>>>(d_A, d_B, d_C, N);
            cudaEventRecord(e3);
            CUDA_CHECK(cudaMemcpy(h_C, d_C, sz, cudaMemcpyDeviceToHost));
            cudaEventRecord(e1); cudaEventSynchronize(e1);
            tiled_kern[s]  = gpu_ms(e2,e3);
            tiled_total[s] = gpu_ms(e0,e1);
            printf("  Tiled CUDA: kern=%.4f ms  total=%.4f ms  correct=%s\n",
                   tiled_kern[s], tiled_total[s], ok(h_ref,h_C,N)?"YES":"NO");
        }

        // cuBLAS
        {
            const float alpha=1.0f, beta=0.0f;
            cudaEventRecord(e0);
            CUDA_CHECK(cudaMemcpy(d_A, h_A, sz, cudaMemcpyHostToDevice));
            CUDA_CHECK(cudaMemcpy(d_B, h_B, sz, cudaMemcpyHostToDevice));
            CUDA_CHECK(cudaMemset(d_C, 0, sz));
            cudaEventRecord(e2);
            CUBLAS_CHECK(cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, N, N,
                                     &alpha, d_B, N, d_A, N, &beta, d_C, N));
            cudaEventRecord(e3);
            CUDA_CHECK(cudaMemcpy(h_C, d_C, sz, cudaMemcpyDeviceToHost));
            cudaEventRecord(e1); cudaEventSynchronize(e1);
            cub_kern[s]  = gpu_ms(e2,e3);
            cub_total[s] = gpu_ms(e0,e1);
            printf("  cuBLAS:     kern=%.4f ms  total=%.4f ms  correct=%s\n",
                   cub_kern[s], cub_total[s], ok(h_ref,h_C,N)?"YES":"NO");
        }

        printf("\n");
        free(h_A); free(h_B); free(h_C); free(h_ref);
        cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
        cudaEventDestroy(e0); cudaEventDestroy(e1);
        cudaEventDestroy(e2); cudaEventDestroy(e3);
    }

    cublasDestroy(handle);

    printf("========================================\n");
    printf("  TABLE 1 - Execution Times (ms)  [tile/block = %dx%d]\n", TS, TS);
    printf("========================================\n");
    printf("%-12s  %-12s  %-18s  %-18s  %-18s  %-18s\n",
           "Matrix", "CPU (ms)", "Basic kern (ms)", "Basic total(ms)", "Tiled kern (ms)", "Tiled total(ms)");
    for (int s = 0; s < 3; s++)
        printf("%-12d  %-12.2f  %-18.4f  %-18.4f  %-18.4f  %-18.4f\n",
               sizes[s], cpu_ms[s], basic_kern[s], basic_total[s], tiled_kern[s], tiled_total[s]);

    printf("\n========================================\n");
    printf("  TABLE 2 - cuBLAS vs CUDA (total end-to-end time + speedup over CPU)\n");
    printf("========================================\n");
    printf("%-12s  %-18s  %-18s  %-18s  %-10s  %-10s  %-10s\n",
           "Matrix", "Basic total(ms)", "Tiled total(ms)", "cuBLAS total(ms)",
           "Spdup Basic", "Spdup Tiled", "Spdup cuBLAS");
    for (int s = 0; s < 3; s++)
        printf("%-12d  %-18.4f  %-18.4f  %-18.4f  %-10.1f  %-10.1f  %-10.1f\n",
               sizes[s], basic_total[s], tiled_total[s], cub_total[s],
               cpu_ms[s]/basic_total[s], cpu_ms[s]/tiled_total[s], cpu_ms[s]/cub_total[s]);

    printf("\n========================================\n");
    printf("  TABLE 3 - GFLOPS (kernel time only)\n");
    printf("========================================\n");
    printf("%-12s  %-14s  %-14s  %-14s\n", "Matrix", "Basic GFLOPS", "Tiled GFLOPS", "cuBLAS GFLOPS");
    for (int s = 0; s < 3; s++) {
        double flops = 2.0 * sizes[s] * sizes[s] * sizes[s];
        printf("%-12d  %-14.2f  %-14.2f  %-14.2f\n", sizes[s],
               flops/(basic_kern[s]*1e6), flops/(tiled_kern[s]*1e6), flops/(cub_kern[s]*1e6));
    }
    printf("\n");
    return 0;
}

In [ ]:
!nvcc -O2 -o benchmark_all 05_benchmark_all.cu -lcublas && ./benchmark_all

---
## Performance Graphs

Run the cell below **after** the benchmark above completes.  
It generates two graphs from the benchmark output:  
1. Matrix size vs execution time (log scale)  
2. Matrix size vs speedup over CPU

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sizes = [256, 512, 1024]

# Paste your actual numbers from TABLE 1 / TABLE 2 output here:
cpu_ms    = [75.34,   598.72,  4782.15]
basic_ms  = [3.45,    9.87,    46.23]
tiled_ms  = [2.93,    6.56,    21.90]
cublas_ms = [2.61,    4.94,    9.78]

# Graph 1: Execution Time vs Matrix Size (log scale)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sizes, cpu_ms,    'o-', label='CPU Sequential',    linewidth=2, markersize=7)
ax.plot(sizes, basic_ms,  's-', label='Basic CUDA',        linewidth=2, markersize=7)
ax.plot(sizes, tiled_ms,  '^-', label='Tiled CUDA (16×16)',linewidth=2, markersize=7)
ax.plot(sizes, cublas_ms, 'D-', label='cuBLAS',            linewidth=2, markersize=7)

ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xticks(sizes)
ax.set_xticklabels([f'{s}×{s}' for s in sizes])
ax.set_xlabel('Matrix Size', fontsize=12)
ax.set_ylabel('Execution Time (ms) — log scale', fontsize=12)
ax.set_title('Matrix Multiplication: Execution Time vs Matrix Size\nAditi Mittal | PRN: 23610067', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('graph1_time_vs_size.png', dpi=150)
plt.show()
print('Saved: graph1_time_vs_size.png')

# Graph 2: Speedup over CPU
basic_spdup  = [cpu_ms[i]/basic_ms[i]  for i in range(3)]
tiled_spdup  = [cpu_ms[i]/tiled_ms[i]  for i in range(3)]
cublas_spdup = [cpu_ms[i]/cublas_ms[i] for i in range(3)]

x = np.arange(len(sizes))
w = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - w, basic_spdup,  w, label='Basic CUDA')
bars2 = ax.bar(x,     tiled_spdup,  w, label='Tiled CUDA (16×16)')
bars3 = ax.bar(x + w, cublas_spdup, w, label='cuBLAS')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.1f}×',
                ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([f'{s}×{s}' for s in sizes])
ax.set_xlabel('Matrix Size', fontsize=12)
ax.set_ylabel('Speedup over CPU', fontsize=12)
ax.set_title('Speedup over CPU Sequential Baseline\nAditi Mittal | PRN: 23610067', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('graph2_speedup_vs_size.png', dpi=150)
plt.show()
print('Saved: graph2_speedup_vs_size.png')

---
## Done!

**What to do next:**
1. Copy the actual timing numbers from the benchmark output into the graph cell above
2. Re-run the graph cell to generate accurate graphs
3. Take screenshots of:
   - `!nvidia-smi` output (GPU model)
   - TABLE 1 and TABLE 2 from `./benchmark_all`
   - Both graphs
4. Add screenshots to your report

**Summary of what was demonstrated:**
- CPU sequential O(N³) → slow, but good cache utilisation with ikj ordering
- Basic CUDA: parallel but hits global memory on every access
- Tiled CUDA: shared-memory tiling reduces global memory accesses by TILE_SIZE×
- cuBLAS: NVIDIA's auto-tuned library, fastest of all — uses register tiling + warp-level tricks